In [1]:
import numpy as np
from drifter import DroguedDrifter
import matplotlib.pyplot as plt
import copernicusmarine

/Users/merlefriederichsen/src/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
x = np.linspace(0, 1000, 100)
y = np.linspace(0, 1000, 100)
z = np.linspace(0, 5, 20)

X, Y, Z = np.meshgrid(x, y, z)

In [3]:
ds_subset = copernicusmarine.open_dataset(dataset_id="cmems_mod_bal_phy_anfc_P1D-m")

ds_subset

INFO - 2026-03-09T13:06:30Z - Selected dataset version: "202411"
INFO - 2026-03-09T13:06:30Z - Selected dataset part: "default"


<xarray.Dataset> Size: 806GB
Dimensions:    (depth: 56, latitude: 774, longitude: 763, time: 1197)
Coordinates:
  * depth      (depth) float32 224B 0.5016 1.516 2.548 ... 663.9 688.0 712.0
  * latitude   (latitude) float32 3kB 53.01 53.02 53.04 ... 65.86 65.87 65.89
  * longitude  (longitude) float32 3kB 9.042 9.069 9.097 ... 30.15 30.18 30.21
  * time       (time) datetime64[ns] 10kB 2022-12-02 2022-12-03 ... 2026-03-12
Data variables:
    bottomT    (time, latitude, longitude) float32 3GB dask.array<chunksize=(100, 774, 763), meta=np.ndarray>
    mlotst     (time, latitude, longitude) float32 3GB dask.array<chunksize=(100, 774, 763), meta=np.ndarray>
    siconc     (time, latitude, longitude) float32 3GB dask.array<chunksize=(100, 774, 763), meta=np.ndarray>
    sithick    (time, latitude, longitude) float32 3GB dask.array<chunksize=(100, 774, 763), meta=np.ndarray>
    so         (time, depth, latitude, longitude) float32 158GB dask.array<chunksize=(100, 2, 774, 763), meta=np.ndarray>
    sob        (time, latitude, longitude) float32 3GB dask.array<chunksize=(100, 774, 763), meta=np.ndarray>
    thetao     (time, depth, latitude, longitude) float32 158GB dask.array<chunksize=(100, 2, 774, 763), meta=np.ndarray>
    uo         (time, depth, latitude, longitude) float32 158GB dask.array<chunksize=(100, 2, 774, 763), meta=np.ndarray>
    vo         (time, depth, latitude, longitude) float32 158GB dask.array<chunksize=(100, 2, 774, 763), meta=np.ndarray>
    wo         (time, depth, latitude, longitude) float32 158GB dask.array<chunksize=(100, 2, 774, 763), meta=np.ndarray>
Attributes: (12/19)
    Conventions:               CF-1.0
    comment:                   Data on cropped native product grid. Horizonta...
    compression:               yes
    contact:                   servicedesk.cmems@mercator-ocean.eu
    creation_date:             2024-11-25 17:05:09
    easternmost_longitude:     30.208656311035156
    ...                        ...
    southernmost_latitude:     53.008296966552734
    start_date:                2024-11-30 12:00:00
    stop_date:                 2024-11-30 12:00:00
    title:                     CMEMS NEMO daily integrated model fields
    westernmost_longitude:     9.041582107543945
    copernicusmarine_version:  2.3.0

In [4]:
def xy_to_lonlat(x_m, y_m, lon0, lat0):
    lat = lat0 + y_m / 111_320.0
    lon = lon0 + x_m / (111_320.0 * np.cos(np.deg2rad(lat0)))
    return lon, lat

In [5]:
def get_uv(
    t,
    z_d,
    y_b,
    x_b,
    ds_subset=None,
    t_ref=np.datetime64("2025-01-01T00:00:00"),
    lon0=20.0,
    lat0=56.0,
):

    lon_b, lat_b = xy_to_lonlat(x_b, y_b, lon0, lat0)
    t_abs = t_ref + np.timedelta64(int(np.rint(t)), "s")

    U_b = float(
        ds_subset["uo"]
        .sel(
            time=t_abs,
            latitude=lat_b,
            longitude=lon_b,
            depth=0.0,
            method="nearest",
        )
        .values
    )

    V_b = float(
        ds_subset["vo"]
        .sel(
            time=t_abs,
            latitude=lat_b,
            longitude=lon_b,
            depth=0.0,
            method="nearest",
        )
        .values
    )

    U_d = float(
        ds_subset["uo"]
        .sel(
            time=t_abs,
            latitude=lat_b,
            longitude=lon_b,
            depth=z_d,
            method="nearest",
        )
        .values
    )

    V_d = float(
        ds_subset["vo"]
        .sel(
            time=t_abs,
            latitude=lat_b,
            longitude=lon_b,
            depth=z_d,
            method="nearest",
        )
        .values
    )

    return U_b, V_b, U_d, V_d

def get_uv(t, z_d, y_b, x_b, ds_subset=None):
    u0 = 0.5
    z_gb = 0
    z_gd = z_d / 5.0

    # Buoy (everything 0)
    psi_dx_b = (
        (1 / 3)
        * np.pi
        * z_gb
        * np.cos(2 * np.pi * y_b / 200)
        * np.sin(2 * np.pi * x_b / 200)
    )
    psi_dy_b = (
        (1 / 3)
        * np.pi
        * z_gb
        * np.sin(2 * np.pi * y_b / 200)
        * np.cos(2 * np.pi * x_b / 200)
    )

    # Drogue
    psi_dx_d = (
        (1 / 3)
        * np.pi
        * z_gd
        * np.cos(2 * np.pi * y_b / 200)
        * np.sin(2 * np.pi * x_b / 200)
    )
    psi_dy_d = (
        (1 / 3)
        * np.pi
        * z_gd
        * np.sin(2 * np.pi * y_b / 200)
        * np.cos(2 * np.pi * x_b / 200)
    )

    U_b = u0
    V_b = 0.0
    U_d = psi_dy_d
    V_d = -u0 - psi_dx_d

    return U_b, V_b, U_d, V_d

drifter = DroguedDrifter(get_uv = get_uv)
t_eval = np.linspace(0, 60, 60)
t_span = (0, 60)
y0 = np.array([100.0, 200.0, 3*np.pi/4, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=float)
sol = drifter.get_full_solution(t_span, y0, t_eval = t_eval)

In [ ]:
def surface_only(x0, y0, U0, V0, T, dt):
    t = np.arange(0.0, T + dt, dt)
    x = x0 + U0 * t
    y = y0 + V0 * t
    return t, x, y


def coupled_traj(drifter, y0, ds_subset=None, T=3600.0, dt=1.0, atol=1e-3, rtol=1e-3):
    t_eval = np.arange(0.0, T + dt, dt)
    sol = drifter.get_full_solution(
        (0.0, T),
        y0,
        ds_subset=ds_subset,
        t_eval=t_eval,
        atol=atol,
        rtol=rtol,
    )

    x_b = sol.y[0, :]
    y_b = sol.y[1, :]

    return sol.t, x_b, y_b, sol


drifter = DroguedDrifter(get_uv=get_uv)
y0 = np.array([100.0, 450.0, 3 * np.pi / 4, 0.0, 0, 0, 0, 0])

T = 3600.0
dt = 10.0

t_s, x_s, y_s = surface_only(y0[0], y0[1], U0=0.5, V0=0, T=T, dt=dt)

# Coupled
t_c, x_b, y_b, sol = coupled_traj(drifter, y0, ds_subset=ds_subset, T=T, dt=dt)

# Plot
plt.figure()
plt.plot(x_s, y_s, label="surface-only")
plt.plot(x_b, y_b, label="coupled system")
plt.scatter([y0[0]], [y0[1]], s=30, label="start", color="k")
plt.axis("equal")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.legend()
plt.title("trajectories: surface-only vs coupled (buoy & drogue)")
plt.show()